## Starting from the EDA I have done...

In [1]:
### Import the dependecies
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [2]:
## Loading the dataset
cc_df = pd.read_csv('../data/raw/creditcard.csv')

In [3]:
# Duplicates
print("Duplicated rows: ", cc_df.duplicated().sum())
cc_df = cc_df.drop_duplicates()

Duplicated rows:  1081


## Feature Engineering

- Feature engineering is the process of transforming or combining raw columns into new variables that make the underlying pattern easier for a model to detect

In [4]:
# I should convert Time (seconds since first transaction) into hour-of-day equivalent
# The dataset spans ~2 days, so Time modulo 86400 (seconds/day) gives a cyclical hour signal
cc_df['hour_of_day'] = (cc_df['Time'] % 86400) // 3600
print(cc_df.groupby('hour_of_day')['Class'].mean())

hour_of_day
0.0     0.000785
1.0     0.002376
2.0     0.014510
3.0     0.004875
4.0     0.010436
5.0     0.003681
6.0     0.002205
7.0     0.003180
8.0     0.000880
9.0     0.001015
10.0    0.000483
11.0    0.003158
12.0    0.001105
13.0    0.001109
14.0    0.001392
15.0    0.001588
16.0    0.001342
17.0    0.001736
18.0    0.001651
19.0    0.001221
20.0    0.001078
21.0    0.000908
22.0    0.000585
23.0    0.001562
Name: Class, dtype: float64


#### One thing to note here
After looking the above data, one can confirm that 11.0 has the highest fraud time of the day. But, due to a large transaction which are not fraud ahve taken place it has diluted its importance. But 2.0 AM has less transaction as there are no transactions made at this time. So, 2.0 isconsidered to be the most fraud transactions taken time


In [5]:
print(cc_df.groupby('hour_of_day')['Class'].sum())


hour_of_day
0.0      6
1.0     10
2.0     48
3.0     17
4.0     23
5.0     11
6.0      9
7.0     23
8.0      9
9.0     16
10.0     8
11.0    53
12.0    17
13.0    17
14.0    23
15.0    26
16.0    22
17.0    28
18.0    28
19.0    19
20.0    18
21.0    16
22.0     9
23.0    17
Name: Class, dtype: int64


In [6]:
# used this to know how many transactions were flagged as fraud and
# 473 were counted as frauds
print(cc_df['Class'].sum())

473


## 3. Data Transformation and Imbalance Handling

### Scaling, Encoding, and SMOTE
 The steps to follow:
1. Use Log-Transfomr on Amount feature first to establish balance
1. Split into train/test (stratified, to preserve class ratio in both)
2. Scale — fit the scaler on train only, then apply to test (same leakage principle applies here too, actually, though slightly less catastrophic)
3. Apply SMOTE/undersampling to the training set only
4. Leave the test set completely untouched in its original, real-world imbalanced form — because that's what production data will actually look like

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Step 1: Log-transform Amount first (heavy right-skew, established earlier)
cc_df['Amount_log'] = np.log1p(cc_df['Amount'])

X_cc = cc_df.drop(columns=['Class', 'Amount'])  # drop raw Amount, keep Amount_log
y_cc = cc_df['Class']

# Step 2: Stratified split
X_cc_train, X_cc_test, y_cc_train, y_cc_test = train_test_split(
    X_cc, y_cc, test_size=0.2, stratify=y_cc, random_state=42
)

print(y_cc_train.value_counts(normalize=True))
print(y_cc_test.value_counts(normalize=True))

# Step 3: Scale: only Time, Amount_log, hour_of_day are meaningfully "raw scale" — 
# V1-V28 are already PCA-standardized by construction, but scaling them again is harmless and standard practice
scaler_cc = StandardScaler()
X_cc_train_scaled = scaler_cc.fit_transform(X_cc_train)
X_cc_test_scaled = scaler_cc.transform(X_cc_test)

# Step 4: SMOTE on train only
print("Before SMOTE:", y_cc_train.value_counts())
smote_cc = SMOTE(random_state=42)
X_cc_train_resampled, y_cc_train_resampled = smote_cc.fit_resample(X_cc_train_scaled, y_cc_train)
print("After SMOTE:", y_cc_train_resampled.value_counts())

Class
0    0.998335
1    0.001665
Name: proportion, dtype: float64
Class
0    0.998326
1    0.001674
Name: proportion, dtype: float64
Before SMOTE: Class
0    226602
1       378
Name: count, dtype: int64
After SMOTE: Class
0    226602
1    226602
Name: count, dtype: int64
